# Variant Study Template

Copy this notebook per variant. Run cells top-to-bottom; sections 3 and 6 require human judgement (track selection / inspection).

Pipeline:
1. Parse SNP input (rsid / GTEx-style / chr:pos:ref:alt) → one-line VCF, and pull its gnomAD annotation (AF / CADD / REVEL / SpliceAI / phylop) from the full observed index.
2. Run `hound_snp_folds` for fast summary stats with gnomad-common quantile normalization.
3. Inspect summary scores — pick interesting tracks (per `targets_df.group`) by quantile rank, with sortable cov + gene-score tables.
4. Re-score with `--stats REF,ALT` for only those tracks.
5. Kick off the slow `hound_ism_snp_folds` in the background — **after** the fast + REF/ALT forward passes, so the long ISM jobs don't clog the GPU queue ahead of them.
6. Visualize ref vs alt with the Plotly browser in gene context; save single-track panels as needed.
7. Once ISM finishes, switch tracks interactively with `ism_track_switcher`.

**Notes:**
- hydra blocks require GPU. Scoring uses the fold scripts' default backend: SLURM when `slurmrunner` is installed, else local (add `--backend gcp` to the commands to use GCP Batch). The notebook process blocks until jobs complete.
- rsid inputs resolve against the full gnomAD observed index (`RSID_INDEX`), so **rare** variants — not just common ones — are found.
- Set `SEQ_LENGTH = 524288` in §0 to shrink the input window from 768 kb to 512 kb so the models fit on smaller GPUs.

## 0. Configuration

In [ ]:
import os, json, shlex, subprocess, time
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from baskerville import dataset
from baskerville.gene import Transcriptome, find_overlapping_genes
from baskerville.snps import compute_score_quantiles
from baskerville.vcf import (
    parse_snp_input,
    write_snp_vcf,
    fetch_gnomad_annotation,
)
from baskerville.viz import (
    plot_snp,
    seq_window_for_variant,
    plot_ism_logo,
    ism_track_switcher,
    launch_ism_bg,
    check_ism,
    plotly_browser_snp,
)

# ---- user inputs -------------------------------------------------------
SNP_INPUT = "chr6_146898162_C_A_b38"  # rsid | GTEx | chr:pos:ref:alt
MODEL_DIR = "/path/to/models/"  # fold dirs f*c0/ plus params.json
PARAMS_FILE = os.path.join(MODEL_DIR, "params.json")
TARGETS_FILE = "/path/to/data/hg38/targets.txt"
GENES_GTF = "/path/to/gencode.gtf"  # hg38 GENCODE gene annotation
HG38_FASTA = "/path/to/hg38.fa"  # with hg38.fa.fai next to it
# -n: subdir under each fold dir holding a scores.h5 from a prior hound_snp_folds
# run over common gnomAD variants, used as the quantile reference (docs/snps.md).
# Set to None and drop '-n' below if you don't have one.
GNOMAD_NORM = "gnomad_common"

# variant resolution + annotation from the full gnomAD observed index:
GNOMAD_DIR = "/path/to/gnomad"  # local gnomAD index (layout below)
RSID_INDEX = os.path.join(
    GNOMAD_DIR, "data/rsid_index.sqlite"
)  # rsid -> pos (incl. rare)
OBSERVED_DIR = os.path.join(GNOMAD_DIR, "data")  # per-chrom observed *.vcf.bgz
RSID_VCF = os.path.join(GNOMAD_DIR, "vcf/common7m.vcf")  # fallback (common only)

# Shrink the model input window to fit smaller GPUs. None keeps the model's
# native seq_length (768 kb); 524288 = 512 kb. Patched params are written per
# variant in §1 and used for all scoring + plot geometry.
SEQ_LENGTH = 524288  # e.g. 524288 for smaller GPUs

SLURM_QUEUE = "gpu"  # GPU queue (or GCP GPU type) for hydra blocks
N_FOLDS_PARALLEL = 8  # max concurrent SLURM jobs
WINDOW_BP = 100_000  # ± around variant for plotting
TOP_PER_GROUP = 3  # how many top tracks per group to surface in Section 3
ISM_MUT_LEN = 128  # centered bp window hound_ism mutates (-l)

OUT_ROOT = Path("./snp_out")

## 1. Parse SNP + write one-line VCF

In [ ]:
# resolve the variant. rsid inputs hit the full observed index first (so rare
# rsids resolve), then fall back to the common-only VCF scan.
snp = parse_snp_input(SNP_INPUT, rsid_index=RSID_INDEX, rsid_vcf=RSID_VCF)
# Resolve the numeric length before choosing a cache directory.
with open(PARAMS_FILE) as f:
    _p = json.load(f)
_m = _p.get("model", _p)
effective_length = int(_m["seq_length"] if SEQ_LENGTH is None else SEQ_LENGTH)
# Key the cache on the full variant: snp.rsid is just 'chr:pos' for colon-form
# inputs, so two alt alleles at one position would otherwise share a directory.
SNP_KEY = f"{snp.chr}_{snp.pos}_{snp.ref_allele}_{snp.alt_allele}"
OUT_DIR = OUT_ROOT / SNP_KEY / f"seq_{effective_length}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
VCF_PATH = str(OUT_DIR / "variant.vcf")
write_snp_vcf(snp, VCF_PATH)
print(snp)
print("VCF:", VCF_PATH)
print("OUT_DIR:", OUT_DIR)

# gnomAD annotation from the full observed index (works for rare variants too)
ann = fetch_gnomad_annotation(
    snp.chr, snp.pos, snp.ref_allele, snp.alt_allele, OBSERVED_DIR
)
if ann is None:
    print("gnomAD: not found in observed index (novel / unobserved)")
else:
    print("gnomAD:", "  ".join(f"{k}={v}" for k, v in ann.items()))

# Effective params: optionally patch seq_length down for smaller GPUs. Used by
# all scoring + seq_window_for_variant so geometry stays consistent.
if SEQ_LENGTH is None:
    EFFECTIVE_PARAMS_FILE = PARAMS_FILE
else:
    from baskerville.viz.snp import _trunk_stride_crop

    _stride, _ = _trunk_stride_crop(_m)
    if effective_length % _stride:
        raise ValueError(
            f"SEQ_LENGTH={SEQ_LENGTH} not divisible by trunk stride {_stride}; "
            f"pick a multiple of {_stride}"
        )
    _m["seq_length"] = effective_length
    EFFECTIVE_PARAMS_FILE = str(OUT_DIR / f"params_{effective_length}.json")
    with open(EFFECTIVE_PARAMS_FILE, "w") as f:
        json.dump(_p, f)
    print(
        f"patched seq_length -> {SEQ_LENGTH} ({SEQ_LENGTH / 1000:.0f} kb): {EFFECTIVE_PARAMS_FILE}"
    )
print("PARAMS  :", EFFECTIVE_PARAMS_FILE)

## 2. Fast SNP scoring (SLURM, synchronous)

Dispatches one GPU job per fold and blocks until the ensemble is built. Takes a few minutes wall-clock once jobs start running.

In [ ]:
SNP_SCORES_DIR = OUT_DIR / "snp_scores"
snp_scores_h5 = SNP_SCORES_DIR / "ensemble" / "scores.h5"
if snp_scores_h5.exists():
    print(f"exists, skipping compute: {snp_scores_h5}")
else:
    cmd = [
        "hound_snp_folds",
        "-f",
        HG38_FASTA,
        "-g",
        GENES_GTF,
        "--rc",
        "--stats",
        "cov/logSUM,cov/logD2,covgene/logFC",
        "-t",
        TARGETS_FILE,
        "-n",
        GNOMAD_NORM,
        "-o",
        str(SNP_SCORES_DIR),
        "-p",
        str(N_FOLDS_PARALLEL),
        "-q",
        SLURM_QUEUE,
        EFFECTIVE_PARAMS_FILE,
        MODEL_DIR,
        VCF_PATH,
    ]
    print(" ".join(shlex.quote(c) for c in cmd))
    t0 = time.time()
    subprocess.run(cmd, check=True)  # blocks until all SLURM jobs complete
    print(f"snp scoring took {time.time() - t0:.0f}s")

## 3. Inspect summary scores, pick interesting tracks

Rank tracks by their quantile vs gnomAD-common. The sortable tables below (cov + gene) let you pick a group and sort by the most interesting effect.

In [ ]:
scores_h5 = h5py.File(SNP_SCORES_DIR / "ensemble" / "scores.h5", "r")
# targets_cov.txt (copied into ensemble/) is the processed target table
# whose row order matches the h5 cov score columns
targets_df = pd.read_csv(
    SNP_SCORES_DIR / "ensemble" / "targets_cov.txt", sep="\t", index_col=0
)

# <stat>_quantiles is per-target thresholds (num_targets, n_q); the matching
# quantile positions are the shared 1-D 'quantiles' dataset
qpos = scores_h5["quantiles"][:]

df = targets_df.copy()
for key in ("cov/logSUM", "cov/logD2"):
    s = scores_h5[key][0].astype("float32")  # (num_targets,) this SNP
    thr = scores_h5[key + "_quantiles"][:]  # (num_targets, n_q)
    df[key] = s
    # helper wants (num_snps, num_targets); we have one SNP
    df[key + "_q"] = compute_score_quantiles(s[None, :], thr, qpos)[0]

# logD2 is an L2 norm (always >= 0); rank by quantile vs gnomad common —
# high q = unusually large effect for this track
df.sort_values("cov/logD2_q", ascending=False).head(20)

In [ ]:
# exclude ChIP-seq (identified by the assay column) and the
# stability / cpa / eclip groups before picking tracks
EXCLUDE_GROUPS = {"stability", "cpa", "eclip"}
assay = df.get("assay", pd.Series("", index=df.index)).fillna("").astype(str)
grp = df["group"].fillna("").astype(str)
keep = ~assay.str.contains("chip", case=False) & ~grp.str.lower().isin(EXCLUDE_GROUPS)
df_sel = df[keep]
print(
    f"excluded {(~keep).sum()} tracks (ChIP-seq assay / {sorted(EXCLUDE_GROUPS)} "
    f"groups); {keep.sum()} remain across {df_sel['group'].nunique()} groups"
)

# top-K tracks per group, ranked by quantile vs gnomad common
top_per_group = (
    df_sel.sort_values("cov/logD2_q", ascending=False)
    .groupby("group", sort=False)
    .head(TOP_PER_GROUP)
    .sort_values(["group", "cov/logD2_q"], ascending=[True, False])
)
top_per_group[
    [
        "identifier",
        "description",
        "group",
        "cov/logSUM",
        "cov/logSUM_q",
        "cov/logD2",
        "cov/logD2_q",
    ]
]

In [ ]:
# Sortable cov-score table: pick a group and sort by the most interesting effect.
# The widgets do the filtering/sorting in Python and re-render a plain pandas
# HTML table -- this renders offline and inside VS Code, unlike itables (whose
# JS loader hangs in VS Code's sandboxed per-cell outputs). Reused for genes.
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


def interactive_score_table(
    frame,
    sort_cols,
    default_sort,
    *,
    group_col="group",
    extra_col=None,
    abs_option=True,
    default_group="gtex",
    max_rows=300,
):
    groups = ["ALL"] + sorted(frame[group_col].dropna().astype(str).unique())
    grp = widgets.Dropdown(
        options=groups,
        value=(default_group if default_group in groups else "ALL"),
        description=group_col,
    )
    sort_dd = widgets.Dropdown(
        options=sort_cols, value=default_sort, description="sort by"
    )
    dir_tb = widgets.ToggleButtons(
        options=[("desc", False), ("asc", True)], value=False, description="order"
    )
    row = [grp, sort_dd, dir_tb]
    abs_cb = widgets.Checkbox(value=False, description="|sort|") if abs_option else None
    if abs_cb is not None:
        row.append(abs_cb)
    extra_dd = None
    if extra_col is not None:
        opts = ["ALL"] + sorted(frame[extra_col].dropna().astype(str).unique())
        extra_dd = widgets.Dropdown(options=opts, value="ALL", description=extra_col)
        row.append(extra_dd)
    out = widgets.Output()

    def _render(*_):
        with out:
            clear_output(wait=True)
            f = frame
            if grp.value != "ALL":
                f = f[f[group_col].astype(str) == grp.value]
            if extra_dd is not None and extra_dd.value != "ALL":
                f = f[f[extra_col].astype(str) == extra_dd.value]
            key = sort_dd.value
            if abs_cb is not None and abs_cb.value:
                f = f.reindex(f[key].abs().sort_values(ascending=dir_tb.value).index)
            else:
                f = f.sort_values(key, ascending=dir_tb.value)
            n = len(f)
            html = f.head(max_rows).to_html(
                index=False,
                float_format=lambda x: f"{x:.4g}",
                border=0,
                classes="dataframe",
            )
            note = (
                f" &nbsp;<i>(top {max_rows} of {n})</i>"
                if n > max_rows
                else f" &nbsp;<i>({n} rows)</i>"
            )
            display(
                HTML(f'<div style="max-height:520px; overflow:auto">{html}</div>{note}')
            )

    for w in row:
        w.observe(_render, names="value")
    _render()
    display(widgets.VBox([widgets.HBox(row), out]))


COV_COLS = [
    "identifier",
    "description",
    "group",
    "cov/logSUM",
    "cov/logSUM_q",
    "cov/logD2",
    "cov/logD2_q",
]
interactive_score_table(
    df_sel[COV_COLS].reset_index(drop=True),
    sort_cols=["cov/logD2_q", "cov/logSUM_q", "cov/logD2", "cov/logSUM"],
    default_sort="cov/logD2_q",
)

In [ ]:
# Sortable gene-specific scores (covgene/logFC), one row per (gene, track).
# covgene is scored over a strand+gene-track SUBSET of the cov targets, written
# to targets_covgene.txt (its index labels are positions into the full cov
# quantile array), so use that table for labels and to slice the quantiles.
covgene_targets_path = SNP_SCORES_DIR / "ensemble" / "targets_covgene.txt"
if "covgene/logFC" not in scores_h5 or not covgene_targets_path.exists():
    print("no covgene/ scores in this run (add covgene/logFC to --stats in §2)")
else:
    tg = pd.read_csv(covgene_targets_path, sep="\t", index_col=0)
    gene_ids = [
        g.decode() if isinstance(g, bytes) else g for g in scores_h5["gene_ids"][:]
    ]
    gene_idx = scores_h5["gene_idx"][:]
    cg = scores_h5["covgene/logFC"][:].astype("float32")  # (n_pairs, n_covgene)
    assert cg.shape[1] == len(tg), (cg.shape, len(tg))
    # quantiles are stored over the full cov target set; select the covgene rows
    cg_thr = scores_h5["covgene/logFC_quantiles"][:][
        tg.index.values
    ]  # (n_covgene, n_q)

    base = tg[["identifier", "description", "group"]].reset_index(drop=True)
    frames = []
    for p in range(cg.shape[0]):
        q = compute_score_quantiles(cg[p][None, :], cg_thr, qpos)[0]
        sub = base.copy()
        sub.insert(0, "gene", gene_ids[int(gene_idx[p])])
        sub["covgene/logFC"] = cg[p]
        sub["covgene/logFC_q"] = q
        frames.append(sub)
    gene_score_df = pd.concat(frames, ignore_index=True)
    print(f"{cg.shape[0]} gene(s) x {len(base)} tracks = {len(gene_score_df)} rows")

    interactive_score_table(
        gene_score_df,
        sort_cols=["covgene/logFC_q", "covgene/logFC"],
        default_sort="covgene/logFC_q",
        extra_col="gene",
    )

## 4. Pick tracks + re-score with REF/ALT (SLURM)

Edit `interesting_track_ids` below based on what jumped out above. We write a sliced targets file so the REF/ALT tensors stay small.

In [ ]:
# default: take the union of top-K per group
interesting_track_ids = top_per_group["identifier"].tolist()

# hound_snp needs a targets file in the ORIGINAL index / strand_pair space.
# targets_cov.txt has collapsed strands + a reset index, so its strand_pair
# values no longer resolve once sliced. Slice the raw targets file instead,
# and keep each selected track's strand_pair partner -- otherwise
# score_snps' strand remap raises KeyError on the missing partner.
targets_full = pd.read_csv(TARGETS_FILE, sep="\t", index_col=0)
sel = targets_full.index[targets_full["identifier"].isin(interesting_track_ids)]
keep_idx = set(sel) | set(targets_full.loc[sel, "strand_pair"])
targets_sub_in = targets_full[targets_full.index.isin(keep_idx)]  # keeps file order

targets_sub_path = OUT_DIR / "targets_subset.txt"
targets_sub_in.to_csv(targets_sub_path, sep="\t")
print(
    f"{len(interesting_track_ids)} datasets -> {len(targets_sub_in)} targets "
    f"(incl. strand partners) -> {targets_sub_path}"
)

In [ ]:
REFALT_DIR = OUT_DIR / "snp_refalt"
refalt_h5 = REFALT_DIR / "ensemble" / "scores.h5"
# delete REFALT_DIR to force a recompute if you changed the track subset above
if refalt_h5.exists():
    print(f"exists, skipping compute: {refalt_h5}")
else:
    cmd = [
        "hound_snp_folds",
        "-f",
        HG38_FASTA,
        "-g",
        GENES_GTF,
        "--rc",
        "--stats",
        "REF,ALT",
        "-t",
        str(targets_sub_path),
        "-o",
        str(REFALT_DIR),
        "-p",
        str(N_FOLDS_PARALLEL),
        "-q",
        SLURM_QUEUE,
        EFFECTIVE_PARAMS_FILE,
        MODEL_DIR,
        VCF_PATH,
    ]
    print(" ".join(shlex.quote(c) for c in cmd))
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f"REF/ALT scoring took {time.time() - t0:.0f}s")

## 5. Kick off ISM (SLURM, background)

Launched **after** the fast (§2) and REF/ALT (§4) forward passes so the multi-hour ISM jobs don't sit ahead of those quick jobs in the GPU queue. Dispatches per-fold GPU jobs and returns immediately. Takes hours; log streams to `OUT_DIR/ism_scores/ism.log`.

In [ ]:
ism_out = OUT_DIR / "ism_scores"
ism_scores_h5 = ism_out / "ensemble" / "scores.h5"
if ism_scores_h5.exists():
    ism_proc, ism_log = None, str(ism_out / "ism.log")
    print(f"exists, skipping compute: {ism_scores_h5}")
else:
    ism_proc, ism_log = launch_ism_bg(
        params_file=EFFECTIVE_PARAMS_FILE,
        models_dir=MODEL_DIR,
        vcf_file=VCF_PATH,
        out_dir=str(ism_out),
        targets_file=TARGETS_FILE,
        genome_fasta=HG38_FASTA,
        genes_gtf=GENES_GTF,
        stats=("cov/logSUM", "cov/logD2", "covgene/logFC"),
        mut_len=ISM_MUT_LEN,
        rc=True,
        parallel=N_FOLDS_PARALLEL,
        queue=SLURM_QUEUE,
    )
    print("ISM PID:", ism_proc.pid)
    print("log    :", ism_log)
    print("status :", check_ism(ism_proc))

## 6. Visualize ref vs alt — Plotly browser

Full-width interactive browser with group + zoom dropdowns (incl. 50 kb) and native pan/zoom/hover, plus a per-zoom gene panel. The cell below it saves a single track as a standalone PNG.

In [ ]:
refalt_h5 = h5py.File(REFALT_DIR / "ensemble" / "scores.h5", "r")
# cov/REF and cov/ALT are raw per-target predictions (NOT strand-collapsed),
# so their target axis matches the input subset file row order -- not the
# strand-collapsed ensemble/targets_cov.txt.
targets_sub = pd.read_csv(
    OUT_DIR / "targets_subset.txt", sep="\t", index_col=0
).reset_index(drop=True)
# shape: (num_snps=1, num_shifts, n_bins, num_targets) -> avg shifts -> (num_targets, n_bins)
ref = np.asarray(refalt_h5["cov/REF"][0].mean(axis=0).T)
alt = np.asarray(refalt_h5["cov/ALT"][0].mean(axis=0).T)
# NOTE: score_snps already applies dataset.untransform_preds before storing
# REF/ALT, so these are already in raw count space -- do NOT untransform again.
print(
    "ref shape:",
    ref.shape,
    "alt shape:",
    alt.shape,
    "(targets_sub rows:",
    len(targets_sub),
    ")",
)

# use EFFECTIVE_PARAMS_FILE so geometry matches the (possibly reduced) seq_length
geom = seq_window_for_variant(EFFECTIVE_PARAMS_FILE, snp.pos)
print("geom:", geom)

tx = Transcriptome(GENES_GTF)
trees = tx.gene_trees()

# full-width Plotly browser: group + zoom dropdowns (incl. 50 kb), native
# pan/zoom/hover. gene_trees+chrom refresh the gene panel per zoom level.
browser = plotly_browser_snp(
    ref,
    alt,
    variant_pos_bp=snp.pos,
    seq_start_bp=geom["seq_start_bp"],
    targets_df=targets_sub,
    bin_size=geom["bin_size"],
    gene_trees=trees,
    chrom=snp.chr,
    same_scale=False,
    show_diff=True,
    title=f"{snp.rsid}  {snp.chr}:{snp.pos} {snp.ref_allele}>{snp.alt_allele}",
)
browser

In [ ]:
# Save a single track's ref/alt coverage as a standalone PNG (static matplotlib,
# publication-ready) -- the clean single-track companion to the browser above.
def save_track_png(t_idx, path=None, *, window_half_bp=WINDOW_BP, show_diff=True, **kw):
    win = (
        (snp.pos - window_half_bp, snp.pos + window_half_bp) if window_half_bp else None
    )
    # nearest gene for context (optional panel)
    gene = None
    if window_half_bp:
        hits = find_overlapping_genes(
            trees, snp.chr, snp.pos - window_half_bp, snp.pos + window_half_bp
        )
        if hits:
            gene = min(
                hits,
                key=lambda g: min(
                    abs(g.span()[0] - snp.pos), abs(g.span()[1] - snp.pos)
                ),
            )
    label = str(targets_sub.iloc[t_idx].get("description", f"track {t_idx}"))
    fig, _ = plot_snp(
        ref,
        alt,
        variant_pos_bp=snp.pos,
        seq_start_bp=geom["seq_start_bp"],
        track_indices=[t_idx],
        targets_df=targets_sub,
        bin_size=geom["bin_size"],
        gene=gene,
        window_bp=win,
        show_diff=show_diff,
        title=f"{snp.rsid}  {label}",
        **kw,
    )
    if path is None:
        path = OUT_DIR / f"track_{targets_sub.iloc[t_idx]['identifier']}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)
    return fig


# example: save the first selected track (edit t_idx for others)
save_track_png(0);

## 7. ISM visualization (run after background job completes)

Re-run the status cell until it reports `done`, then load and visualize.

In [ ]:
print(
    "ISM output present \u2014 launch was skipped"
    if ism_proc is None
    else check_ism(ism_proc)
)

In [ ]:
ism_h5 = h5py.File(OUT_DIR / "ism_scores" / "ensemble" / "scores.h5", "r")

# seqs are stored (4, mut_len); scores (mut_len, 4, num_targets)
ref_seq = ism_h5["ref/seqs"][0].T  # -> (mut_len, 4)
alt_seq = ism_h5["alt/seqs"][0].T
variant_offset = ref_seq.shape[0] // 2  # variant sits at the window center

ISM_STATS = ["logD2", "logSUM"]  # both available; switchable in the UI
scores = {}
for stat in ISM_STATS:
    r = ism_h5[f"ref/cov/{stat}"][0].astype("float32")  # (mut_len, 4, num_targets)
    a = ism_h5[f"alt/cov/{stat}"][0].astype("float32")
    # mean-normalize across nucleotides per position (matches viz_ism.ipynb).
    # logD2 is a divergence (non-negative-ish); ADD mean/3 so the wild-type
    # nucleotide column lifts up to ~0 instead of zeroing out. logSUM is
    # signed; subtract the mean to center the four bases at zero.
    if stat == "logD2":
        r += r.mean(axis=1, keepdims=True) / 3
        a += a.mean(axis=1, keepdims=True) / 3
    else:
        r -= r.mean(axis=1, keepdims=True)
        a -= a.mean(axis=1, keepdims=True)
    scores[stat] = (r, a)

ism_track_switcher(
    targets_df=targets_df,
    ref_seq=ref_seq,
    alt_seq=alt_seq,
    scores=scores,
    variant_offset=variant_offset,
)